# 99 · Validate checkpoints
Asserts every number in the master checkpoint table (learning/README) and the KPI checkpoints. Run after a full pipeline run.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
b, s, g = f"{catalog}.bronze", f"{catalog}.silver", f"{catalog}.gold"

def scalar(sql):
    return spark.sql(sql).first()[0]

CHECKS = {
    # Bronze (latest batch counts equal totals on a first load)
    "bronze.per_all_people_f":          (f"SELECT COUNT(*) FROM {b}.per_all_people_f", 2701),
    "bronze.per_all_assignments_m":     (f"SELECT COUNT(*) FROM {b}.per_all_assignments_m", 2720),
    "bronze.per_periods_of_service":    (f"SELECT COUNT(*) FROM {b}.per_periods_of_service", 2693),
    "bronze.learning_activity":         (f"SELECT COUNT(*) FROM {b}.learning_activity", 17928),
    "bronze.hr_all_organization_units": (f"SELECT COUNT(*) FROM {b}.hr_all_organization_units", 40),
    "bronze.per_jobs":                  (f"SELECT COUNT(*) FROM {b}.per_jobs", 60),
    "bronze.per_grades_f":              (f"SELECT COUNT(*) FROM {b}.per_grades_f", 12),
    "bronze.per_locations_f":           (f"SELECT COUNT(*) FROM {b}.per_locations_f", 10),
    "bronze.per_legal_employers":       (f"SELECT COUNT(*) FROM {b}.per_legal_employers", 4),
    # Silver
    "silver.hcmworker (current)":       (f"SELECT COUNT(*) FROM {s}.hcmworker WHERE IsCurrent", 2693),
    "silver.hcmworker dob missing":     (f"SELECT COUNT(*) FROM {s}.hcmworker WHERE IsCurrent AND IsDobMissing", 20),
    "silver.hcmassignment":             (f"SELECT COUNT(*) FROM {s}.hcmassignment", 2705),
    "silver.hcmassignment primary":     (f"SELECT COUNT(*) FROM {s}.hcmassignment WHERE IsPrimary", 2693),
    "silver.hcmperiodsofservice":       (f"SELECT COUNT(*) FROM {s}.hcmperiodsofservice", 2688),
    "silver.periods terminated":        (f"SELECT COUNT(*) FROM {s}.hcmperiodsofservice WHERE IsTerminated", 688),
    "silver.hcmjob (current)":          (f"SELECT COUNT(*) FROM {s}.hcmjob WHERE IsCurrent", 60),
    "silver.hcmgrade (current)":        (f"SELECT COUNT(*) FROM {s}.hcmgrade WHERE IsCurrent", 12),
    "silver.hcmlearningactivity":       (f"SELECT COUNT(*) FROM {s}.hcmlearningactivity", 17923),
    # Gold
    "gold.dimdate":                     (f"SELECT COUNT(*) FROM {g}.dimdate", 6574),
    "gold.dimworker":                   (f"SELECT COUNT(*) FROM {g}.dimworker", 2693),
    "gold.dimorganization":             (f"SELECT COUNT(*) FROM {g}.dimorganization", 40),
    "gold.dimjob":                      (f"SELECT COUNT(*) FROM {g}.dimjob", 60),
    "gold.dimgrade":                    (f"SELECT COUNT(*) FROM {g}.dimgrade", 12),
    "gold.dimlocation":                 (f"SELECT COUNT(*) FROM {g}.dimlocation", 11),
    "gold.dimlegalentity":              (f"SELECT COUNT(*) FROM {g}.dimlegalentity", 5),
    "gold.factworkforceheadcount":      (f"SELECT COUNT(*) FROM {g}.factworkforceheadcount", 116999),
    "gold.factworkforceattrition":      (f"SELECT COUNT(*) FROM {g}.factworkforceattrition", 688),
    "gold.factlearningactivity":        (f"SELECT COUNT(*) FROM {g}.factlearningactivity", 17923),
    # KPIs
    "KPI headcount 2026-06-30":         (f"SELECT SUM(PrimaryHeadcount) FROM {g}.factworkforceheadcount WHERE SnapshotDateKey = 20260630", 2005),
    "KPI voluntary leavers":            (f"SELECT SUM(VoluntaryAttritionCount) FROM {g}.factworkforceattrition", 492),
    "KPI learning completions":         (f"SELECT SUM(CompletionCount) FROM {g}.factlearningactivity", 13999),
}

In [ ]:
rows = []
for name, (sql, expected) in CHECKS.items():
    actual = scalar(sql)
    rows.append((name, expected, actual, "✅" if actual == expected else "❌"))
display(spark.createDataFrame(rows, "check STRING, expected LONG, actual LONG, status STRING"))

In [ ]:
hours = scalar(f"SELECT CAST(SUM(Hours) AS STRING) FROM {g}.factlearningactivity")
attr_rate = scalar(f"""
WITH hc AS (
  SELECT AVG(m) avg_hc FROM (
    SELECT f.SnapshotDateKey, SUM(f.PrimaryHeadcount) m
    FROM {g}.factworkforceheadcount f JOIN {g}.dimdate d ON d.DateKey = f.SnapshotDateKey
    WHERE d.Year = 2025 GROUP BY 1)),
lv AS (
  SELECT SUM(AttritionCount) leavers FROM {g}.factworkforceattrition a
  JOIN {g}.dimdate d ON d.DateKey = a.TerminationDateKey WHERE d.Year = 2025)
SELECT ROUND(100 * leavers / avg_hc, 1) FROM hc, lv""")
compliance = scalar(f"""
SELECT ROUND(100 * SUM(CASE WHEN IsMandatory AND IsCompleted THEN 1 END) / SUM(MandatoryTrainingCount), 1)
FROM {g}.factlearningactivity""")
print(f"Learning hours {hours} (expect 90585.60) | 2025 attrition {attr_rate}% (expect 14.4) | "
      f"mandatory compliance {compliance}% (expect 81.4)")

failed = [r[0] for r in rows if r[3] == "❌"]
if hours != "90585.60":
    failed.append("learning hours")
if float(attr_rate) != 14.4:
    failed.append("2025 attrition rate")
if float(compliance) != 81.4:
    failed.append("mandatory compliance")
assert not failed, f"Checkpoints failed: {failed}"
print("All checkpoints match.")